# Demo A: Why Serving an LLM Is Hard — Measured, Start to Finish

**ODSC 2026** | Serving LLMs Efficiently at Scale | [Harshul Jain](https://github.com/harshuljain13)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/odsc-2026/demos/odsc_2026_demo_a_model_memory_prefill_decode.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/workshop/odsc-2026/demos/odsc_2026_demo_a_model_memory_prefill_decode.ipynb)

## One chain of reasoning

Each section exists because the previous one raised a question:

```
  §1  We download a model. It is N GB of weights.
        └─ what else needs memory, and how much is left?
  §2  Weights are a FIXED cost. The KV cache is the PER-USER one.
        └─ what does a user actually experience?
  §3  TTFT across 5 short and 4 long prompts. It tracks input length, closely.
        └─ where inside that latency did the time go?
  §4  Prefill scales with the prompt; decode is flat and far slower per token.
        └─ if prefill is the expensive part of a long prompt, can we skip it?
  §5  Sometimes. Long prompts that SHARE A PREFIX let you reuse the cache and
      skip most of prefill. Dissimilar prompts of identical length cannot.
        └─ and why is decode able to be "flat" at all?
  §6  The KV cache. Measured over enough steps for the O(n) growth to show.
        └─ flat, but still slow. What is decode waiting on?
  §7  Memory bandwidth: every token re-reads every weight. Measured, not quoted.
        └─ so what does that cache weigh?
  §8  Weigh it. It matches §2's prediction — size deployments from config.json.
        └─ what does all this cost in money?
  §9  Output tokens are decode, so they are billed 4-5x higher. Our unbatched
      ratio is far worse than that, and the gap is batching.
        └─ floor = (weights + KV) / bandwidth. Shrink both → Demo B.
```

## How the measurements are taken

Benchmarking a GPU badly is easy and the results look confident anyway. This
notebook enforces three rules, in the `repeat()` harness in §0:

1. **Synchronize** around every timed region — otherwise you time a launch queue.
2. **Warm up** first — the first call compiles kernels and allocates peaks.
3. **Repeat and report the median with its range** — a single sample is an anecdote,
   and every table below carries `n` and a low–high spread so you can see the noise.

Every number is measured on the machine running this notebook. The one value we
*predict* from `config.json` (KV bytes per token, §2) is labelled a prediction and
checked against the real cache in §8.

**Hardware:** CUDA with ≥ 18 GB runs the 7B path; anything smaller or CPU-only falls
back to a 1.1B model and smaller knobs.

## §0 · Setup

Hardware, knobs, and three helper cells — plots, benchmark harness, model ops. After
this the notebook is experiments only.

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers', 'torch', 'matplotlib', 'pandas', 'accelerate'])

import copy
import gc
import time

import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device('cuda' if HAS_CUDA else 'cpu')

if HAS_CUDA:
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Device : {GPU_NAME} ({VRAM_GB:.1f} GB VRAM)')
else:
    GPU_NAME, VRAM_GB = 'CPU', 0.0
    print('Device : CPU (no CUDA). Slower, but every effect still shows.')
print(f'torch  : {torch.__version__}')

In [ ]:
# --- Knobs ----------------------------------------------------------------
BIG, SMALL = 'mistralai/Mistral-7B-v0.1', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
USE_BIG = HAS_CUDA and VRAM_GB >= 18

MODEL_ID = BIG if USE_BIG else SMALL
DTYPE = torch.float16 if HAS_CUDA else torch.float32

REPEATS = 5 if HAS_CUDA else 3          # timed repetitions per measurement
DECODE_STEPS = 16                       # steps for the per-phase comparison
GROWTH_STEPS = 64 if HAS_CUDA else 16   # steps for the cache O(n) test (§6)
LONG_TOKENS = [512, 1024, 2048, 4096] if USE_BIG else (
    [256, 512, 1024] if HAS_CUDA else [128, 256])
PREFIX_TOKENS = 2048 if USE_BIG else (512 if HAS_CUDA else 128)
N_REQUESTS = 4                          # requests per family in §5

print(f'Model        : {MODEL_ID}')
print(f'Precision    : {str(DTYPE).replace("torch.", "")}')
print(f'Repeats      : {REPEATS} (median reported, range shown)')
print(f'Long prompts : {LONG_TOKENS} tokens')
print(f'Shared prefix: {PREFIX_TOKENS} tokens | {N_REQUESTS} requests per family')

In [ ]:
# --- Plot helpers ---------------------------------------------------------
# Every chart in this notebook is one call into these. Keeps the story cells
# readable instead of burying the point under axis bookkeeping.
PALETTE = ['#dbeafe', '#dcfce7', '#fef3c7', '#ffedd5', '#ffe4e6', '#f3e8ff']
COOL, WARM = '#2563eb', '#991b1b'


def panels(n=1, width=6.5, height=4.5):
    fig, axes = plt.subplots(1, n, figsize=(width * n, height))
    return (axes,) if n == 1 else axes


def _finish(ax, title, xlabel=None, ylabel=None, legend=False):
    ax.set_title(title, fontsize=12, fontweight='bold')
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=11)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=11)
    if legend:
        ax.legend(fontsize=9)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)


def bars(ax, labels, values, title, ylabel, fmt='{:.0f}'):
    values = list(values)
    ax.bar(range(len(values)), values, color=PALETTE[:len(values)],
           edgecolor='#000', linewidth=1.2)
    ax.set_xticks(range(len(values)))
    ax.set_xticklabels(labels)
    for i, v in enumerate(values):
        ax.text(i, v, fmt.format(v), ha='center', va='bottom', fontsize=10)
    ax.set_ylim(0, max(values) * 1.25)
    _finish(ax, title, ylabel=ylabel)


def hbars(ax, labels, values, title, xlabel, notes=None, fmt='{:.2f}'):
    values = list(values)
    ax.barh(range(len(values)), values, color=PALETTE[:len(values)],
            edgecolor='#000', linewidth=1.2)
    ax.set_yticks(range(len(values)))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    for i, v in enumerate(values):
        label = fmt.format(v) + (f'  ({notes[i]})' if notes else '')
        ax.text(v + max(values) * 0.02, i, label, va='center', fontsize=9)
    ax.set_xlim(0, max(values) * 1.4)
    _finish(ax, title, xlabel=xlabel)


def stacked(ax, labels, parts, title, ylabel, fmt=None):
    # parts: [(series_name, [values aligned to labels]), ...]
    bottom = [0.0] * len(labels)
    for i, (name, vals) in enumerate(parts):
        ax.bar(labels, vals, bottom=bottom, color=PALETTE[i % len(PALETTE)],
               edgecolor='#000', linewidth=1.2, label=name)
        bottom = [b + v for b, v in zip(bottom, vals)]
    if fmt:
        for i, total in enumerate(bottom):
            ax.text(i, total, fmt.format(total), ha='center', va='bottom', fontsize=9)
    _finish(ax, title, ylabel=ylabel, legend=True)


def grouped(ax, groups, series, title, ylabel, fmt='{:.1f}'):
    # series: [(series_name, [values aligned to groups]), ...]
    width = 0.8 / len(series)
    for si, (name, vals) in enumerate(series):
        xs = [g + si * width - 0.4 + width / 2 for g in range(len(groups))]
        ax.bar(xs, vals, width, color=PALETTE[si % len(PALETTE)],
               edgecolor='#000', linewidth=1.1, label=name)
        for x, v in zip(xs, vals):
            ax.text(x, v, fmt.format(v), ha='center', va='bottom', fontsize=9)
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(groups)
    _finish(ax, title, ylabel=ylabel, legend=True)


def curves(ax, series, title, xlabel, ylabel, logy=False, zero_floor=True):
    # series: [(name, xs, ys, color_or_None), ...]
    for i, (name, xs, ys, color) in enumerate(series):
        ax.plot(xs, ys, marker='o', markersize=3.5, linewidth=1.8,
                color=color or [COOL, WARM][i % 2], label=name)
    if logy:
        ax.set_yscale('log')
    elif zero_floor:
        ax.set_ylim(bottom=0)
    _finish(ax, title, xlabel=xlabel, ylabel=ylabel, legend=len(series) > 1)


print('Plot helpers ready: bars, hbars, stacked, grouped, curves, panels.')

def points(ax, xs, ys, title, xlabel, ylabel, lo=None, hi=None, label=None,
           color=None, logy=False):
    # Median with a min-max range bar. Use when a number is a measurement, not a fact.
    yerr = None
    if lo is not None and hi is not None:
        yerr = [[y - l for y, l in zip(ys, lo)], [h - y for y, h in zip(ys, hi)]]
    ax.errorbar(xs, ys, yerr=yerr, fmt='o-', capsize=4, linewidth=1.8, markersize=5,
                color=color or COOL, label=label)
    if logy:
        ax.set_yscale('log')
    else:
        ax.set_ylim(bottom=0)
    _finish(ax, title, xlabel=xlabel, ylabel=ylabel, legend=label is not None)


def spread(ax, groups, samples, title, ylabel):
    # One box per group — shows the distribution instead of hiding it in a mean.
    style = dict(patch_artist=True,
                 boxprops=dict(facecolor=PALETTE[0], edgecolor='#000'),
                 medianprops=dict(color='#000', linewidth=1.5))
    try:                                      # matplotlib >= 3.9
        ax.boxplot(samples, tick_labels=groups, **style)
    except TypeError:                         # older matplotlib
        ax.boxplot(samples, labels=groups, **style)
    ax.set_ylim(bottom=0)
    _finish(ax, title, ylabel=ylabel)



In [ ]:
# --- Benchmark harness ----------------------------------------------------
# Three rules this harness enforces, because getting them wrong produces
# confident nonsense:
#   1. synchronize the device around every timed region (else you time a queue)
#   2. warm up before measuring (the first call compiles kernels)
#   3. repeat and report the MEDIAN with its range (one sample is an anecdote)
from dataclasses import dataclass
from typing import List, Optional


@dataclass
class Stat:
    """Summary of repeated timings, in milliseconds."""
    median: float
    low: float
    high: float
    n: int

    @staticmethod
    def of(seconds: List[float]) -> 'Stat':
        ms = sorted(s * 1000 for s in seconds)
        mid = ms[len(ms) // 2] if len(ms) % 2 else (ms[len(ms) // 2 - 1] + ms[len(ms) // 2]) / 2
        return Stat(median=mid, low=ms[0], high=ms[-1], n=len(ms))

    def __str__(self):
        return f'{self.median:.2f} ms [{self.low:.2f}-{self.high:.2f}, n={self.n}]'


@dataclass
class Prompt:
    """A tokenized prompt plus the metadata we group results by."""
    name: str
    kind: str                     # 'short' | 'long'
    ids: 'torch.Tensor'

    @property
    def n_tokens(self) -> int:
        return int(self.ids.shape[1])


def sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


def time_once(fn):
    """Wall time of fn(), device-synchronized on both sides."""
    sync()
    t0 = time.perf_counter()
    result = fn()
    sync()
    return result, time.perf_counter() - t0


def repeat(fn, n=REPEATS, warmup=1) -> Stat:
    """Warm up, then time fn() n times and summarize."""
    for _ in range(warmup):
        fn()
    return Stat.of([time_once(fn)[1] for _ in range(n)])


def free_memory():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


print(f'Benchmark harness ready (median of {REPEATS}, with warmup).')


## §1 · We Download a Model. How Big Is It?

The two numbers the rest of the notebook depends on: **parameter count** and **bytes on
the device**.

In [ ]:
config = AutoConfig.from_pretrained(MODEL_ID)
N_LAYERS = config.num_hidden_layers
N_HEADS = config.num_attention_heads
N_KV_HEADS = getattr(config, 'num_key_value_heads', None) or N_HEADS
HEAD_DIM = getattr(config, 'head_dim', None) or (config.hidden_size // N_HEADS)

print(f'Layers       : {N_LAYERS}')
print(f'Hidden dim   : {config.hidden_size}')
print(f'Q / KV heads : {N_HEADS} / {N_KV_HEADS}' +
      ('   <- GQA: the KV side is narrower' if N_KV_HEADS < N_HEADS else '   (no GQA)'))
print(f'Head dim     : {HEAD_DIM}')

In [ ]:
print(f'Loading {MODEL_ID} ...')

model, load_sec = None, None
t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map='auto' if HAS_CUDA else None).eval()
if not HAS_CUDA:
    model.to(DEVICE)
load_sec = time.perf_counter() - t0

N_PARAMS = sum(p.numel() for p in model.parameters())
WEIGHT_BYTES = sum(p.numel() * p.element_size() for p in model.parameters())

print(f'Load time    : {load_sec:.1f} s')
print(f'Parameters   : {N_PARAMS/1e9:.2f} B at {WEIGHT_BYTES/N_PARAMS:.1f} bytes each')
print(f'WEIGHTS      : {WEIGHT_BYTES/1e9:.2f} GB   <- carried to §7')

In [ ]:
# --- Model-level operations ----------------------------------------------
# Each function does exactly one thing to the model, so the story cells below
# read as a sequence of experiments rather than a wall of torch calls.


def tokenize(text: str, max_tokens: Optional[int] = None) -> 'torch.Tensor':
    kwargs = dict(return_tensors='pt')
    if max_tokens:
        kwargs.update(truncation=True, max_length=max_tokens)
    return tokenizer(text, **kwargs)['input_ids'].to(DEVICE)


def build_prompt(text_source: str, target_tokens: int) -> 'torch.Tensor':
    """Repeat text_source until it reaches exactly target_tokens."""
    repeats = max(1, target_tokens // max(len(tokenize(text_source)[0]), 1) + 1)
    return tokenize(text_source * repeats, max_tokens=target_tokens)


@torch.no_grad()
def prefill(ids, keep_cache=True, past=None, attention_mask=None):
    """One forward pass over `ids`. Returns (outputs, seconds).

    This IS time-to-first-token: the first output token falls out of this pass.
    """
    call = lambda: model(input_ids=ids, past_key_values=past,
                         attention_mask=attention_mask, use_cache=keep_cache)
    return time_once(call)


@torch.no_grad()
def decode_cached(prompt_ids, steps) -> List[float]:
    """Decode `steps` tokens the normal way: 1 token in, KV cache reused.

    Returns per-step seconds.
    """
    out, _ = prefill(prompt_ids)
    past, tok = out.past_key_values, pick_next(out)

    per_step = []
    for _ in range(steps):
        out, sec = time_once(lambda: model(input_ids=tok, past_key_values=past,
                                           use_cache=True))
        per_step.append(sec)
        past, tok = out.past_key_values, pick_next(out)
    return per_step


@torch.no_grad()
def decode_uncached(prompt_ids, steps) -> List[float]:
    """Decode `steps` tokens with NO cache: every step re-forwards the whole
    sequence. Returns per-step seconds.
    """
    out, _ = prefill(prompt_ids, keep_cache=False)
    seq, tok = prompt_ids, pick_next(out)

    per_step = []
    for _ in range(steps):
        seq = torch.cat([seq, tok], dim=1)
        out, sec = time_once(lambda: model(input_ids=seq, use_cache=False))
        per_step.append(sec)
        tok = pick_next(out)
    return per_step


def pick_next(out):
    return out.logits[:, -1, :].argmax(dim=-1, keepdim=True)


# transformers has shipped several KV cache representations (legacy tuples,
# key_cache/value_cache lists, newer .layers objects). Normalize to (k, v) pairs.
def cache_layers(past):
    if past is None:
        return []
    if hasattr(past, 'to_legacy_cache'):
        try:
            legacy = past.to_legacy_cache()
            if legacy:
                return [(lay[0], lay[1]) for lay in legacy]
        except Exception:
            pass
    if hasattr(past, 'key_cache') and hasattr(past, 'value_cache'):
        return list(zip(past.key_cache, past.value_cache))
    if hasattr(past, 'layers'):
        try:
            return [(lay.keys, lay.values) for lay in past.layers]
        except Exception:
            pass
    if isinstance(past, (list, tuple)):
        return [(lay[0], lay[1]) for lay in past if lay is not None and len(lay) >= 2]
    return []


def cache_bytes(past):
    return sum(k.numel() * k.element_size() + v.numel() * v.element_size()
               for k, v in cache_layers(past))


def cache_tokens(past):
    if past is not None and hasattr(past, 'get_seq_length'):
        try:
            return int(past.get_seq_length())
        except Exception:
            pass
    lays = cache_layers(past)
    return int(lays[0][0].shape[-2]) if lays else 0


print('Model ops ready: prefill, decode_cached, decode_uncached, cache_bytes.')


## §2 · Fixed Cost vs Per-User Cost

Weights are paid once and shared by everyone. The **KV cache** is per-user and grows with
every token. That distinction decides how many people fit on one GPU.

In [ ]:
def weight_breakdown(model) -> dict:
    """Total bytes per component. Priority matters: lm_head before attn/mlp."""
    RULES = [('LM head', ('lm_head',)),
             ('Embeddings', ('embed_tokens', 'wte', 'word_embeddings', 'embeddings')),
             ('Norms', ('norm', 'ln_')),
             ('Attention (Q,K,V,O)', ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'attn')),
             ('MLP / FFN', ('mlp', 'gate_proj', 'up_proj', 'down_proj', 'fc1', 'fc2'))]

    totals = {}
    for name, p in model.named_parameters():
        low = name.lower()
        part = next((b for b, keys in RULES if any(k in low for k in keys)), 'Other')
        totals[part] = totals.get(part, 0) + p.numel() * p.element_size()
    return dict(sorted(totals.items(), key=lambda kv: -kv[1]))


WEIGHTS_BY_PART = weight_breakdown(model)
pd.DataFrame([{'Component': part, 'Memory (GB)': round(b / 1e9, 3),
               'Share': f'{b/WEIGHT_BYTES*100:.1f}%'}
              for part, b in WEIGHTS_BY_PART.items()])

In [ ]:
if HAS_CUDA:
    sync()
    FREE_B, TOTAL_B = torch.cuda.mem_get_info()
    print(f'Weight tensors : {WEIGHT_BYTES/1e9:.2f} GB')
    print(f'torch allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')
    print(f'Free VRAM      : {FREE_B/1e9:.2f} GB of {TOTAL_B/1e9:.2f} GB')
    print('\nEvery per-user byte — KV cache, activations, fragmentation — lives there.')
else:
    FREE_B = TOTAL_B = 0
    print(f'Weights in host RAM: {WEIGHT_BYTES/1e9:.2f} GB (CPU run)')

In [ ]:
ax1, ax2 = panels(2)

hbars(ax1, list(WEIGHTS_BY_PART), [b / 1e9 for b in WEIGHTS_BY_PART.values()],
      f'The {N_PARAMS/1e9:.1f}B weights, by component', 'GB', fmt='{:.2f} GB')

if HAS_CUDA:
    stacked(ax2, ['GPU'], [('Weights + overhead', [(TOTAL_B - FREE_B) / 1e9]),
                           ('Free for users', [FREE_B / 1e9])],
            'Budget after loading\n(zero users served)', 'VRAM (GB)', fmt='{:.0f} GB')
else:
    ax2.axis('off')
    ax2.text(0.5, 0.5, 'CPU run —\nno VRAM budget', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

### The per-user cost, predicted

$$\text{bytes per token} = 2 \times L \times n_{kv} \times d_h \times \text{bytes per element}$$

Leading 2 = one K plus one V. Note `n_kv`, **not** `n_heads` — that substitution is what
GQA buys, and it is Demo B's opening move. This is the only unmeasured number in the
notebook; §8 weighs the real cache and checks it.

In [ ]:
ELEM_BYTES = torch.finfo(DTYPE).bits // 8
KV_PER_TOKEN_PREDICTED = 2 * N_LAYERS * N_KV_HEADS * HEAD_DIM * ELEM_BYTES

print('PREDICTION (settled in §8)')
print(f'  2 x {N_LAYERS} layers x {N_KV_HEADS} kv-heads x {HEAD_DIM} dim x {ELEM_BYTES} B'
      f'  =  {KV_PER_TOKEN_PREDICTED/1024:.0f} KB per token')
print(f'  a user at {max(LONG_TOKENS)} tokens of context -> '
      f'{KV_PER_TOKEN_PREDICTED*max(LONG_TOKENS)/1e9:.2f} GB')

## §3 · TTFT Across Many Prompts, Not Two

Time to first token is the pause before anything appears. Measuring it on one short and
one long prompt tells you almost nothing — per-call overhead dominates short prompts, and
a single sample hides the noise.

So: **five short prompts and four long ones**, each timed `REPEATS` times, median with
range reported.

In [ ]:
# Five genuinely different short prompts, and four long ones built from
# distinct source texts so "long" is not one prompt measured four times.
SHORT_TEXTS = [
    'Explain quantum computing in simple terms:',
    'Write a haiku about distributed systems.',
    'What causes inflation?',
    'Summarize the plot of Hamlet in one sentence.',
    'List three ways to reduce cloud spend:',
]

LONG_SOURCES = [
    'The transformer architecture processes every token in parallel during the '
    'forward pass, which makes prompt ingestion fast but generation sequential. ',
    'Distributed storage systems trade consistency for availability under network '
    'partitions, a tension formalized by the CAP theorem and refined by PACELC. ',
    'Photosynthesis converts light energy into chemical energy stored in glucose, '
    'using water as an electron donor and releasing oxygen as a byproduct. ',
    'Monetary policy transmits to the real economy through interest rates, asset '
    'prices, credit availability, and exchange rate channels with variable lags. ',
    'Plate tectonics describes the slow motion of lithospheric plates over the '
    'asthenosphere, driving earthquakes, mountain building, and seafloor spreading. ',
]
# LONG_SOURCES[0] is reserved as the shared prefix in §5; [1:] are the
# mutually-distinct texts used for the dissimilar family.

PROMPTS = [Prompt(f'short-{i+1}', 'short', tokenize(t))
           for i, t in enumerate(SHORT_TEXTS)]
PROMPTS += [Prompt(f'long-{n}', 'long', build_prompt(LONG_SOURCES[i % len(LONG_SOURCES)], n))
            for i, n in enumerate(LONG_TOKENS)]

for p in PROMPTS:
    print(f'  {p.name:<10} {p.kind:<6} {p.n_tokens:>5} tokens')

In [ ]:
# Warm up once on the longest prompt: the first call pays for kernel
# compilation and peak allocation, and timing that would be timing the wrong thing.
longest = max(PROMPTS, key=lambda p: p.n_tokens)
prefill(longest.ids)
free_memory()

ttft = {p.name: repeat(lambda p=p: prefill(p.ids)) for p in PROMPTS}

ttft_table = pd.DataFrame([{
    'Prompt': p.name, 'Kind': p.kind, 'Input tokens': p.n_tokens,
    'TTFT median (ms)': round(ttft[p.name].median, 2),
    'TTFT low (ms)': round(ttft[p.name].low, 2),
    'TTFT high (ms)': round(ttft[p.name].high, 2),
    'ms per input token': round(ttft[p.name].median / p.n_tokens, 4),
    'n': ttft[p.name].n,
} for p in PROMPTS])
ttft_table

In [ ]:
ax1, ax2 = panels(2)

points(ax1, [p.n_tokens for p in PROMPTS],
       [ttft[p.name].median for p in PROMPTS],
       'TTFT tracks input length', 'Input tokens', 'TTFT (ms)',
       lo=[ttft[p.name].low for p in PROMPTS],
       hi=[ttft[p.name].high for p in PROMPTS])

spread(ax2, ['short prompts', 'long prompts'],
       [[ttft[p.name].median for p in PROMPTS if p.kind == k] for k in ('short', 'long')],
       'Spread within each class', 'TTFT (ms)')
plt.tight_layout()
plt.show()

shorts = [ttft[p.name].median for p in PROMPTS if p.kind == 'short']
longs = [ttft[p.name].median for p in PROMPTS if p.kind == 'long']
print(f'short prompts: {min(shorts):.1f}-{max(shorts):.1f} ms TTFT')
print(f'long prompts : {min(longs):.1f}-{max(longs):.1f} ms TTFT')
print(f'\nNote the "ms per input token" column: it FALLS as prompts get longer.')
print('Short prompts are dominated by fixed per-call overhead, not by their tokens.')
print('That is why one short prompt is a bad measurement of prefill throughput.')

**Where we are:** TTFT is a real, user-visible cost that scales with input length. But
TTFT is only the first token — where does the rest of the time go?

## §4 · Prefill vs Decode

Two phases, driven by hand so they can be timed apart:

| Phase | The call | Tokens per pass | Produces |
|-------|----------|-----------------|----------|
| **Prefill** | `prefill(ids)` — one pass | all of them, in parallel | the first token (= TTFT) |
| **Decode** | `decode_cached(ids, steps)` — one pass per token | exactly 1 | every later token |

Measured for all nine prompts, so the trend is a trend and not two points.

In [ ]:
phase = {}
for p in PROMPTS:
    steps = decode_cached(p.ids, DECODE_STEPS)
    phase[p.name] = Stat.of(steps)
    free_memory()

phase_table = pd.DataFrame([{
    'Prompt': p.name, 'Kind': p.kind, 'Input tokens': p.n_tokens,
    'Prefill (ms)': round(ttft[p.name].median, 2),
    'Prefill (tok/s)': round(p.n_tokens / (ttft[p.name].median / 1000)),
    'Decode (ms/token)': round(phase[p.name].median, 2),
    'Decode (tok/s)': round(1000 / phase[p.name].median, 1),
    'Decode spread (ms)': f'{phase[p.name].low:.2f}-{phase[p.name].high:.2f}',
} for p in PROMPTS])
phase_table

In [ ]:
xs = [p.n_tokens for p in PROMPTS]
ax1, ax2 = panels(2)

points(ax1, xs, [ttft[p.name].median for p in PROMPTS],
       'PREFILL grows with the prompt', 'Input tokens', 'ms per prefill')
points(ax2, xs, [phase[p.name].median for p in PROMPTS],
       'DECODE barely moves', 'Input tokens', 'ms per decoded token',
       lo=[phase[p.name].low for p in PROMPTS],
       hi=[phase[p.name].high for p in PROMPTS], color=WARM)
ax2.set_ylim(0, max(phase[p.name].high for p in PROMPTS) * 1.6)
plt.tight_layout()
plt.show()

fastest_prefill = max(p.n_tokens / (ttft[p.name].median / 1000) for p in PROMPTS)
typical_decode = sorted(1000 / phase[p.name].median for p in PROMPTS)[len(PROMPTS) // 2]
print(f'Best prefill throughput : {fastest_prefill:>9,.0f} tokens/s')
print(f'Median decode throughput: {typical_decode:>9,.1f} tokens/s')
print(f'Prefill is ~{fastest_prefill/typical_decode:,.0f}x more productive per second.')
print('\nSame weights, same GPU. The difference is one parallel pass versus')
print('one pass per token.')

**Where we are:** for a long prompt, prefill is a large, length-proportional cost paid
before the user sees anything. Which raises an obvious question for anyone running a
production service: **do we have to pay it every time?**

## §5 · Similar Prompts vs Dissimilar Prompts

Real traffic is not random. A RAG service sends the same retrieved document with
different questions; an agent replays a long system prompt every turn; a chatbot resends
the conversation so far. Those requests **share a prefix**.

If the prefix is shared, its K and V are identical — so they can be computed once and
reused, and prefill only has to process the part that is new. If the prompts are
genuinely different, there is nothing to reuse.

We measure both families at **identical total input length**:

| Family | Construction | Reuse available? |
|--------|--------------|------------------|
| **similar** | one shared prefix + a different short suffix per request | yes — prefix K/V computed once |
| **dissimilar** | a different source text per request | no — every token is new |

This is the mechanism behind prefix caching in vLLM and RadixAttention in SGLang.

In [ ]:
SUFFIXES = ['\n\nQuestion: what is the main idea?',
            '\n\nQuestion: list the key risks.',
            '\n\nQuestion: summarize in one line.',
            '\n\nQuestion: who would disagree, and why?']


def make_similar_family(n_requests, prefix_tokens):
    """One shared prefix, n different suffixes."""
    prefix = build_prompt(LONG_SOURCES[0], prefix_tokens)
    suffixes = [tokenize(SUFFIXES[i % len(SUFFIXES)]) for i in range(n_requests)]
    return prefix, suffixes


def make_dissimilar_family(suffixes, prefix_tokens):
    """Unrelated prompts, each matching its similar counterpart TOTAL length.

    Sources come from LONG_SOURCES[1:], so none of them shares the prefix text.
    """
    return [build_prompt(LONG_SOURCES[1 + i % (len(LONG_SOURCES) - 1)],
                         prefix_tokens + int(s.shape[1]))
            for i, s in enumerate(suffixes)]


shared_prefix, suffixes = make_similar_family(N_REQUESTS, PREFIX_TOKENS)
dissimilar = make_dissimilar_family(suffixes, PREFIX_TOKENS)

print(f'similar    : {shared_prefix.shape[1]} shared + '
      f'{[int(s.shape[1]) for s in suffixes]} unique = '
      f'{[shared_prefix.shape[1] + int(s.shape[1]) for s in suffixes]} total')
print(f'dissimilar : {[int(d.shape[1]) for d in dissimilar]} total, zero shared prefix')
print('\nTotals match request-for-request, so input length is controlled and the')
print('only variable left is whether the prefix is shared.')

In [ ]:
@torch.no_grad()
def prefill_cold(ids) -> float:
    """Full prefill, nothing reused. Seconds."""
    return repeat(lambda: prefill(ids)).median


@torch.no_grad()
def prefill_with_prefix(prefix_cache, prefix_len, suffix_ids) -> float:
    """Prefill only the NEW tokens, reusing a cached prefix. Median ms.

    The cache is deep-copied per repetition because a forward pass appends to it.
    Only the forward pass is timed — a real engine shares cache blocks
    copy-on-write rather than copying, so the copy is an artifact of this demo.
    """
    total_len = prefix_len + int(suffix_ids.shape[1])
    mask = torch.ones((1, total_len), dtype=torch.long, device=DEVICE)

    def one():
        cache = copy.deepcopy(prefix_cache)
        return prefill(suffix_ids, past=cache, attention_mask=mask)

    return repeat(one).median


# Compute the shared prefix ONCE — this is the cost a prefix cache amortizes.
prefix_out, prefix_build_sec = prefill(shared_prefix)
PREFIX_CACHE = prefix_out.past_key_values
PREFIX_LEN = cache_tokens(PREFIX_CACHE)
print(f'Built the shared prefix cache once: {prefix_build_sec*1000:.1f} ms, '
      f'{PREFIX_LEN} tokens, {cache_bytes(PREFIX_CACHE)/1e6:.1f} MB')

In [ ]:
rows = []
for i, suffix in enumerate(suffixes):
    cold = prefill_cold(torch.cat([shared_prefix, suffix], dim=1))
    warm = prefill_with_prefix(PREFIX_CACHE, PREFIX_LEN, suffix)
    rows.append({'Family': 'similar', 'Request': i + 1,
                 'Input tokens': PREFIX_LEN + int(suffix.shape[1]),
                 'New tokens': int(suffix.shape[1]),
                 'Cold prefill (ms)': round(cold, 2),
                 'With prefix reuse (ms)': round(warm, 2),
                 'Speedup': round(cold / warm, 1)})
    free_memory()

for i, ids in enumerate(dissimilar):
    cold = prefill_cold(ids)
    # Nothing to reuse: these prompts share no meaningful prefix, so the only
    # honest number is the cold one. We report the attempt as 1.0x.
    rows.append({'Family': 'dissimilar', 'Request': i + 1,
                 'Input tokens': int(ids.shape[1]), 'New tokens': int(ids.shape[1]),
                 'Cold prefill (ms)': round(cold, 2),
                 'With prefix reuse (ms)': round(cold, 2), 'Speedup': 1.0})
    free_memory()

prefix_table = pd.DataFrame(rows)
prefix_table

In [ ]:
sim = prefix_table[prefix_table['Family'] == 'similar']
dis = prefix_table[prefix_table['Family'] == 'dissimilar']

ax1, ax2 = panels(2)

grouped(ax1, [f'req {i}' for i in sim['Request']],
        [('cold prefill', list(sim['Cold prefill (ms)'])),
         ('prefix reused', list(sim['With prefix reuse (ms)']))],
        'SIMILAR prompts: most of prefill is skippable', 'ms', fmt='{:.0f}')

grouped(ax2, [f'req {i}' for i in dis['Request']],
        [('cold prefill', list(dis['Cold prefill (ms)'])),
         ('nothing to reuse', list(dis['With prefix reuse (ms)']))],
        'DISSIMILAR prompts: nothing to skip', 'ms', fmt='{:.0f}')
ax2.set_ylim(ax1.get_ylim())      # same scale, so the contrast is honest
plt.tight_layout()
plt.show()

print(f"Similar family   : {sim['Cold prefill (ms)'].mean():7.1f} ms cold -> "
      f"{sim['With prefix reuse (ms)'].mean():6.1f} ms reused "
      f"({sim['Speedup'].mean():.1f}x faster TTFT)")
print(f"Dissimilar family: {dis['Cold prefill (ms)'].mean():7.1f} ms cold -> no reuse possible")
print('\nIdentical input length. Identical model. The only difference is whether the')
print('traffic shares a prefix — and it changes TTFT by the factor above.')
print('\nTwo consequences for anyone running this in production:')
print('  1. your TTFT depends on your TRAFFIC PATTERN, not just your prompt size')
print('  2. routing matters: sending similar requests to the same replica keeps the')
print('     shared prefix warm. Round-robin throws that away.')

**Where we are:** prefill can sometimes be skipped. Decode never can — every token
needs its own pass. And §4 showed those passes cost roughly the same regardless of how
much context exists. Why doesn't attention over 4096 tokens cost more than over 20?

## §6 · The KV Cache, Tested Where It Actually Shows

Because K and V for earlier tokens are cached. Without that, every step would redo the
whole sequence.

**This test has to be designed carefully.** The naive version — take a 4096-token prompt,
decode 12 steps with and without the cache, compare first step to last — detects nothing,
because the sequence grows by 12/4096 ≈ 0.3% and the growth vanishes into noise.

To see O(n) behaviour you need the sequence to grow *substantially* during the test. So we
start from a **short** prompt and decode `GROWTH_STEPS` tokens: the sequence multiplies
several times over, and the uncached cost climbs visibly with it.

In [ ]:
growth_prompt = next(p for p in PROMPTS if p.kind == 'short')
print(f'Starting from {growth_prompt.name} ({growth_prompt.n_tokens} tokens), '
      f'decoding {GROWTH_STEPS} steps.')
print(f'Sequence will grow {growth_prompt.n_tokens} -> '
      f'{growth_prompt.n_tokens + GROWTH_STEPS} tokens '
      f'({(growth_prompt.n_tokens + GROWTH_STEPS)/growth_prompt.n_tokens:.1f}x).\n')

cached_steps = decode_cached(growth_prompt.ids, GROWTH_STEPS)
free_memory()
uncached_steps = decode_uncached(growth_prompt.ids, GROWTH_STEPS)
free_memory()

cached_ms = [s * 1000 for s in cached_steps]
uncached_ms = [s * 1000 for s in uncached_steps]


def trend(ms):
    """Mean of the last quarter vs the first quarter — does cost grow?"""
    q = max(len(ms) // 4, 1)
    first, last = sum(ms[:q]) / q, sum(ms[-q:]) / q
    return first, last, last / first


c_first, c_last, c_ratio = trend(cached_ms)
u_first, u_last, u_ratio = trend(uncached_ms)

growth_table = pd.DataFrame([
    {'Mode': 'with cache', 'First quarter (ms/tok)': round(c_first, 2),
     'Last quarter (ms/tok)': round(c_last, 2), 'Growth': f'{c_ratio:.2f}x',
     'Mean (ms/tok)': round(sum(cached_ms) / len(cached_ms), 2)},
    {'Mode': 'no cache', 'First quarter (ms/tok)': round(u_first, 2),
     'Last quarter (ms/tok)': round(u_last, 2), 'Growth': f'{u_ratio:.2f}x',
     'Mean (ms/tok)': round(sum(uncached_ms) / len(uncached_ms), 2)},
])
growth_table

In [ ]:
ax1, ax2 = panels(2)

curves(ax1, [('with cache', range(1, len(cached_ms) + 1), cached_ms, COOL),
             ('no cache', range(1, len(uncached_ms) + 1), uncached_ms, WARM)],
       'Cost per token as the sequence grows', 'Decode step', 'ms per token')

bars(ax2, ['with cache', 'no cache'],
     [sum(cached_ms) / len(cached_ms), sum(uncached_ms) / len(uncached_ms)],
     f'Mean over {GROWTH_STEPS} steps', 'ms per token', '{:.1f}')
plt.tight_layout()
plt.show()

print(f'With cache: {c_ratio:.2f}x growth across the run — flat, O(1) per token.')
print(f'No cache  : {u_ratio:.2f}x growth across the run — climbing, O(n) per token.')
print('\nThe cache converts work-per-token from "redo the whole sequence" into')
print('"process one token". That is what makes decode flat.')
print('\nCaveat worth stating: on a LONG prompt this comparison looks much weaker,')
print('because adding a few dozen tokens to 4096 barely changes the sequence length.')
print('The cache still saves the same absolute work — the test just cannot see it.')

**Where we are:** decode is flat, and we know why. But flat is not cheap — §4 put decode
orders of magnitude below prefill in tokens/second. What is each step actually waiting
on?

## §7 · The Floor: One Full Weight Read per Token

Hypothesis that explains flat-and-slow at once: **producing one token requires reading
every weight out of memory.** Same cost every step (flat), and large (slow).

If so, decode has a floor set by bandwidth, and faster arithmetic cannot beat it. We
measure this machine's bandwidth instead of quoting a spec sheet, which would be wrong on
any other GPU.

In [ ]:
def measure_bandwidth_gbs(iters=20) -> Optional[float]:
    """Large device-to-device copy: each pass reads nbytes and writes nbytes."""
    if not HAS_CUDA:
        return None
    free_now, _ = torch.cuda.mem_get_info()
    n_elem = max(int(min(512e6, free_now * 0.15)) // 2, 1 << 20)
    try:
        src = torch.empty(n_elem, dtype=torch.float16, device='cuda')
        dst = torch.empty_like(src)
    except torch.cuda.OutOfMemoryError:
        return None

    best_sec = repeat(lambda: dst.copy_(src), n=iters).low / 1000
    moved = 2 * src.numel() * src.element_size()
    del src, dst
    free_memory()
    return moved / best_sec / 1e9


BW_GBS = measure_bandwidth_gbs()
print(f'Measured bandwidth: {BW_GBS:,.0f} GB/s'
      if BW_GBS else 'Bandwidth test skipped (needs CUDA).')
if BW_GBS:
    print('(Copy benchmark — an achievable floor, below the vendor peak figure.)')

In [ ]:
if BW_GBS:
    floor_rows = []
    for p in PROMPTS:
        # KV term uses the §2 PREDICTION (settled in §8). The weight term
        # dominates by orders of magnitude, so the floor barely moves either way.
        kv_b = KV_PER_TOKEN_PREDICTED * (p.n_tokens + DECODE_STEPS)
        floor_ms = (WEIGHT_BYTES + kv_b) / (BW_GBS * 1e9) * 1000
        floor_rows.append({'Prompt': p.name, 'Input tokens': p.n_tokens,
                           'Floor (ms/token)': round(floor_ms, 2),
                           'Measured (ms/token)': round(phase[p.name].median, 2),
                           'Efficiency': f'{floor_ms/phase[p.name].median*100:.0f}%'})
    floor_table = pd.DataFrame(floor_rows)
    display(floor_table)

    ax, = panels(1, width=10)
    grouped(ax, [p.name for p in PROMPTS],
            [('floor: one weight + KV read', list(floor_table['Floor (ms/token)'])),
             ('measured', list(floor_table['Measured (ms/token)']))],
            f'Decode is bandwidth-bound ({BW_GBS:,.0f} GB/s measured)',
            'ms per token', '{:.1f}')
    plt.tight_layout()
    plt.show()

    print(f'Reading {WEIGHT_BYTES/1e9:.1f} GB at {BW_GBS:,.0f} GB/s = '
          f'{WEIGHT_BYTES/(BW_GBS*1e9)*1000:.1f} ms, once per token.')
    print('The gap to measured is launch overhead, attention, sampling, Python —')
    print('all fixable. The floor is not.')
else:
    floor_table = None

## §8 · Weigh the Cache, Settle the §2 Prediction

§2 predicted KV bytes per token from `config.json`. Now weigh real caches at several
context lengths and check.

In [ ]:
kv_rows = []
for p in PROMPTS:
    out, _ = prefill(p.ids)
    measured = cache_bytes(out.past_key_values) / max(cache_tokens(out.past_key_values), 1)
    kv_rows.append({'Prompt': p.name, 'Input tokens': p.n_tokens,
                    'Cache (MB)': round(cache_bytes(out.past_key_values) / 1e6, 2),
                    'Measured B/token': round(measured),
                    'Predicted B/token': KV_PER_TOKEN_PREDICTED,
                    'Error': f'{abs(measured-KV_PER_TOKEN_PREDICTED)/KV_PER_TOKEN_PREDICTED*100:.1f}%'})
    del out
    free_memory()

kv_table = pd.DataFrame(kv_rows)
kv_table

In [ ]:
ax1, ax2 = panels(2)

grouped(ax1, [p.name for p in PROMPTS],
        [('predicted (§2)', [v / 1024 for v in kv_table['Predicted B/token']]),
         ('measured', [v / 1024 for v in kv_table['Measured B/token']])],
        'Prediction vs measurement', 'KB per cached token', '{:.0f}')

points(ax2, list(kv_table['Input tokens']), list(kv_table['Cache (MB)']),
       'One user\'s cache grows linearly with context', 'Context (tokens)', 'MB')
plt.tight_layout()
plt.show()

worst = kv_table['Error'].max()
print(f'Worst prediction error across {len(kv_table)} prompts: {worst}')
print('Small error: capacity planning is arithmetic on config.json.')
print('Large error: suspect sliding-window, MLA, or a quantized cache, and go')
print('read the model code rather than trusting the formula.')

## §9 · Why the API Bill Is Shaped Like This

Same asymmetry, in dollars. Every provider bills input and output tokens differently:

| Our phase | Billed as | Why |
|-----------|-----------|-----|
| **Prefill** | input tokens | one parallel pass, thousands of tokens/second |
| **Decode** | output tokens | one pass per token, each re-reading the weights |

The *direction* of that price gap comes from the hardware. Its *size* does not — and the
comparison below is the most useful surprise in the notebook.

In [ ]:
# List prices, $ per 1M tokens. EXTERNAL FACTS WITH A SHELF LIFE — verify before
# presenting. The durable part is the ratio.
PRICES = {'gpt-4o': {'input': 2.50, 'output': 10.00},
          'gpt-4o-mini': {'input': 0.15, 'output': 0.60},
          'claude-sonnet': {'input': 3.00, 'output': 15.00}}

pd.DataFrame([{'Model': k, 'Input ($/1M)': v['input'], 'Output ($/1M)': v['output'],
               'Output / Input': f"{v['output']/v['input']:.0f}x"}
              for k, v in PRICES.items()])

In [ ]:
# Our measured per-token cost ratio, on the longest prompt (where prefill is
# efficient and the comparison is least flattered by per-call overhead).
biggest = max(PROMPTS, key=lambda p: p.n_tokens)
prefill_ms_per_token = ttft[biggest.name].median / biggest.n_tokens
decode_ms_per_token = phase[biggest.name].median
MEASURED_RATIO = decode_ms_per_token / prefill_ms_per_token
charged = [v['output'] / v['input'] for v in PRICES.values()]

print(f'MEASURED on {GPU_NAME}, {biggest.name} ({biggest.n_tokens} tokens):')
print(f'  input token  : {prefill_ms_per_token:8.4f} ms  (prefill, amortized)')
print(f'  output token : {decode_ms_per_token:8.4f} ms  (decode)')
print(f'  ratio        : {MEASURED_RATIO:8.0f}x\n')
print(f'CHARGED by providers: {", ".join(f"{c:.0f}x" for c in charged)}')
print(f'\nUnbatched we measure ~{MEASURED_RATIO:.0f}x; you are charged ~'
      f'{sum(charged)/len(charged):.0f}x. Those are nowhere near each other,')
print('and the gap is the point:')
print('  - we decoded for ONE user, so one weight read produced one token')
print('  - a real server decodes for hundreds at once, and that SAME read')
print('    produces a token for every user in the batch')
print('\nHardware sets the direction of the price list. BATCHING sets its magnitude.')

In [ ]:
REQS_PER_DAY, AVG_IN, AVG_OUT = 10_000, 500, 200
p = PRICES['gpt-4o']

in_usd = REQS_PER_DAY * AVG_IN * p['input'] / 1e6
out_usd = REQS_PER_DAY * AVG_OUT * p['output'] / 1e6
tok_share = AVG_OUT / (AVG_IN + AVG_OUT) * 100
usd_share = out_usd / (in_usd + out_usd) * 100

print(f'{REQS_PER_DAY:,} req/day x ({AVG_IN} in + {AVG_OUT} out), gpt-4o list price:')
print(f'  daily   ${in_usd + out_usd:,.2f}  (input ${in_usd:,.2f} + output ${out_usd:,.2f})')
print(f'  monthly ${(in_usd + out_usd) * 30:,.2f}')
print(f'\nOutput is {tok_share:.0f}% of tokens but {usd_share:.0f}% of the bill.')

ax1, ax2 = panels(2)
bars(ax1, list(PRICES) + ['OURS\n(1 user,\nunbatched)'], charged + [MEASURED_RATIO],
     'Output-vs-input: charged, then measured', 'ratio (log scale)', '{:.0f}x')
ax1.set_yscale('log')
ax1.set_ylim(1, MEASURED_RATIO * 3)

stacked(ax2, ['Share of tokens', 'Share of dollars'],
        [('Input (prefill)', [100 - tok_share, 100 - usd_share]),
         ('Output (decode)', [tok_share, usd_share])],
        'The bill inverts the token mix', '%')
plt.tight_layout()
plt.show()

## §10 · The Chain, Closed

| Step | Measured | Consequence |
|------|----------|-------------|
| §1 | Weights = N GB on device | The fixed cost of being open for business |
| §2 | Weights fixed, KV cache per-user-per-token | Capacity is a KV question, not a weights question |
| §3 | TTFT across 9 prompts, with spread | TTFT tracks input length; short prompts are overhead-dominated |
| §4 | Prefill scales with prompt; decode flat and far slower per token | Two phases, two bottlenecks, never one average |
| §5 | Shared-prefix prompts skip most of prefill; dissimilar ones cannot | **TTFT depends on your traffic pattern, and routing is part of your latency** |
| §6 | Uncached decode grows per step, cached does not | The cache turns O(n) per token into O(1) |
| §7 | Decode ≈ one full weight read per token | A hard floor from bandwidth, not compute |
| §8 | Real cache size matches the §2 formula | Size deployments from `config.json` |
| §9 | Measured output:input time ratio ≫ the price ratio | Hardware sets direction; **batching** sets magnitude |

### Three numbers to leave with

1. **prefill tok/s ÷ decode tok/s** — the asymmetry behind every batching decision.
2. **weight bytes ÷ measured bandwidth** — your per-token floor.
3. **your prefix hit rate** — how much of your prefill is skippable at all (§5).

### What Demo B does with this

Floor = `(weights + KV) / bandwidth`. The denominator needs a new GPU, so attack the
numerator from both ends — shrink the KV cache (GQA/MQA/MLA) and shrink the weights
(INT8/INT4). Demo B measures both.

### Honest limits

- One GPU, one model, **unbatched** pure PyTorch. Every absolute number here is beaten by
  a real engine; the shapes are what transfer.
- §5 deep-copies the prefix cache per repetition and times only the forward pass. A real
  engine shares cache blocks copy-on-write, so the copy is an artifact of the demo, not a
  cost you would pay.
- §9's prices are external facts with a shelf life. Re-check them; the ratio is durable,
  the dollars are not.
- **Batching is the one major lever this notebook never measures** — and §9 shows it is
  worth more than everything else here combined.